#### PS-s5e8 | [Binary Classification with a Bank Dataset](https://www.kaggle.com/competitions/playground-series-s5e8/code?competitionId=91719&sortBy=scoreDescending&excludeNonAccessedDatasources=true)

| &nbsp; | &nbsp; | &nbsp; | | &nbsp; | &nbsp; |
| -: | - | :-: | :- | :-: | :-: |
|  |  |  |  |  |  |
| [1](#submit) | 0.97772 | &nbsp; v.8 &nbsp; | [PS-S5E8 . Blend XGB+LGB+More](https://www.kaggle.com/code/haohuanchen/ps-s5e8-blend-xgb-lgb-more) |  &nbsp; contributor &nbsp; | [HaohuanChen](https://www.kaggle.com/haohuanchen) |
| [2](#submit) | 0.97770 | &nbsp; v.16 &nbsp; | [🎰 LUCKY SPIN dataset](https://www.kaggle.com/code/antonoof/lucky-spin-dataset) |  &nbsp; expert &nbsp; | [Antonoof](https://www.kaggle.com/antonoof) | 
| [3](#submit) | 0.97768 | &nbsp; v.26 &nbsp; | [LLM is worth trying](https://www.kaggle.com/code/lyf5201133/llm-is-worth-trying) |  &nbsp; contributor &nbsp; | [lyf5201133](https://www.kaggle.com/lyf5201133) |

In [1]:
import ast
import numpy as np
import pandas as pd

In [2]:
def v_blend(path_to_ds, file_short_names, dk):

    def read(dk,i):
        tnm = dk["subm"][i]["name"]
        FiN = dk["path"] + tnm + ".csv"
        return pd.read_csv(FiN).rename(columns={'target':tnm, dk["target"]:tnm})
        
    def merge(dfs_subm):
        df_subms = pd.merge(dfs_subm[0],  dfs_subm[1], on=[dk['id']])
        for i in range(2, len(dk["subm"])): 
            df_subms = pd.merge(df_subms, dfs_subm[i], on=[dk['id']])
        return df_subms
        
    def da(dk,sorting_direction):
        
        df_subms = merge([read(dk,i) for i in range(len(dk["subm"]))])
        cols = [col for col in df_subms.columns if col != dk['id']]
        short_name_cols = [c for c in cols]
        
        def alls(x, sd=sorting_direction,cs=cols):
            reverse = True if sd=='desc' else False
            tes = {c: x[c] for c in cs}.items()
            subms_sorted = [t[0] for t in sorted(tes,key=lambda k:k[1],reverse=reverse)]
            return subms_sorted
            
        def summa(x,cs,wts,ic_alls): 
            return sum([x[cs[j]] * (wts[0][j] + wts[1][ic_alls[j]]) for j in range(len(cs))])
            
        wts = [
            [[e['weight'] for e in dk["subm"]], [w for w in dk["subwts" ]]],
            [[e['weight'] for e in dk["subm2"]],[w for w in dk["subwts2"]]],
            [[e['weight'] for e in dk["subm3"]],[w for w in dk["subwts3"]]],
        ]
        def correct(x, cs=cols, wts=wts):
            i = [x['alls'].index(c) for c in short_name_cols]
            if   0.00 < x['mx-m'] <= 0.10: return summa(x,cs,wts[0],i)
            if   0.10 < x['mx-m'] <= 0.20: return summa(x,cs,wts[1],i)
            else:                          return summa(x,cs,wts[2],i)
                
        def amxm(x, cs=cols):
            list_values = x[cs].to_list()
            mxm = abs(max(list_values)-min(list_values))
            return mxm
            
        df_subms['mx-m']       = df_subms.apply(lambda x: amxm   (x), axis=1)
        df_subms['alls']       = df_subms.apply(lambda x: alls   (x), axis=1)
        df_subms[dk["target"]] = df_subms.apply(lambda x: correct(x), axis=1)
        schema_rename = { old_nc:new_shnc for old_nc, new_shnc in zip(cols, short_name_cols) }
        df_subms = df_subms.rename(columns=schema_rename)
        df_subms = df_subms.rename(columns={dk["target"]:"ensemble"})
        df_subms.insert(loc=1, column=' _ ', value=['   '] * len(df_subms))
        df_subms[' _ '] = df_subms[' _ '].astype(str)
        pd.set_option('display.max_rows',100)
        pd.set_option('display.float_format', '{:.4f}'.format)
        vcols = [dk['id']] + [' _ '] + short_name_cols + [' _ '] + ['mx-m'] + [' _ '] + ['alls'] + [' _ '] + ['ensemble']
        df_subms = df_subms[vcols]
        display(df_subms.head(5))
        pd.set_option('display.float_format', '{:.5f}'.format)
        df_subms = df_subms.rename(columns={"ensemble":dk["target"]})
        df_subms.to_csv(f'tida_{sorting_direction}.csv', index=False)
        return df_subms[[dk['id'],dk['target']]]
   
    def ensemble_da(dk): 
        dfD = da(dk,'desc')
        dfA = da(dk,'asc')
        dfA[dk['target']] = dk['desc'] * dfD[dk['target']] + \
                            dk['asc']  * dfA[dk['target']]
        return dfA
    
    return  ensemble_da(dk)


def data_in_col(i,matrix):
    data = [row[i] for row in matrix]
    return data

def quantity(i,js):
    return {"c" : i, "q" : sum(1 for subm in cols[i] if subm == subms[js])}

def dossier(js):
    return {
        'name' : subms[js],
        'q_in' : [quantity(i,js) for i in range(len(subms))]
    }

def info_mx_m(df):
    matrix = [ast.literal_eval(row.alls) for row in df.itertuples()]
    subms = sorted(matrix[0])
    df_subms = pd.DataFrame({f'col_{i}': data_in_col(i,matrix) for i in range(len(subms))})
    fig,(ax1,ax2,ax3) = plt.subplots(ncols=5,figsize=(12,3))
    axs = [ax1,ax2,ax3]
    for i in range(len(subms)):
        axs[i] = sns.countplot(x=df_subms[f"col_{i}"],ax=axs[i])
    plt.tight_layout()
    dossiers = [dossier(js) for js in range(len(subms))]
    for one_dossier in dossiers: 
        print(one_dossier['name'])
        for q in one_dossier['q_in']:
            print("\t",q)


## submit

In [3]:
%%time

path ='/kaggle/input/30-august-2025-ps-s5e8/' + 'submission '

fins = ['0.97772','0.97770','0.97768']

params_v16 = {
        'path'   : path,
        'id'     : 'id',
        'target' : "y",
        'desc'   : 0.70,
        'asc'    : 0.30,
        'subwts' : [+0.03, -0.02, -0.01],
        'subm'   : [
             { 'name':fins[0],'weight':0.85, },
             { 'name':fins[1],'weight':0.14, },
             { 'name':fins[2],'weight':0.01, },
        ],
        'subwts2': [+0.04, -0.01, -0.03],
        'subm2'  : [
             { 'name':fins[0],'weight':0.79, },
             { 'name':fins[1],'weight':0.19, },            # LB = ?
             { 'name':fins[2],'weight':0.02, },
        ],
        'subwts3': [+0.03, -0.01, -0.02],
        'subm3'  : [
             { 'name':fins[0],'weight':0.85, },
             { 'name':fins[1],'weight':0.14, },
             { 'name':fins[2],'weight':0.01, },
        ]
    }

params_v17 = {
        'path'   : path,
        'id'     : 'id',
        'target' : "y",
        'desc'   : 0.70,
        'asc'    : 0.30,
        'subwts' : [-0.01, -0.02, +0.03],
        'subm'   : [
             { 'name':fins[0],'weight':0.85, },
             { 'name':fins[1],'weight':0.14, },
             { 'name':fins[2],'weight':0.01, },
        ],
        'subwts2': [-0.03, -0.01, +0.04],
        'subm2'  : [
             { 'name':fins[0],'weight':0.79, },
             { 'name':fins[1],'weight':0.19, },            # LB = ?
             { 'name':fins[2],'weight':0.02, },
        ],
        'subwts3': [-0.02, -0.01, +0.03],
        'subm3'  : [
             { 'name':fins[0],'weight':0.85, },
             { 'name':fins[1],'weight':0.14, },
             { 'name':fins[2],'weight':0.01, },
        ]
    }


params = params_v16


df = v_blend ( path, fins, params )

df.to_csv('submission.csv', index=False)

display(df)

,id,_,0.97772,0.97770,0.97768,_,mx-m,_,alls,_,ensemble
0,750000,,0.0072,0.0006,0.0108,,0.0102,,"[0.97768, 0.97772, 0.97770]",,0.0065
1,750001,,0.0765,0.0120,0.0757,,0.0645,,"[0.97772, 0.97768, 0.97770]",,0.0682
2,750002,,0.0007,0.0001,0.0011,,0.0010,,"[0.97768, 0.97772, 0.97770]",,0.0007
3,750003,,0.0006,0.0001,0.0010,,0.0009,,"[0.97768, 0.97772, 0.97770]",,0.0006
4,750004,,0.0091,0.0012,0.0107,,0.0095,,"[0.97768, 0.97772, 0.97770]",,0.0082


,id,_,0.97772,0.97770,0.97768,_,mx-m,_,alls,_,ensemble
0,750000,,0.0072,0.0006,0.0108,,0.0102,,"[0.97770, 0.97772, 0.97768]",,0.0061
1,750001,,0.0765,0.0120,0.0757,,0.0645,,"[0.97770, 0.97768, 0.97772]",,0.0656
2,750002,,0.0007,0.0001,0.0011,,0.0010,,"[0.97770, 0.97772, 0.97768]",,0.0006
3,750003,,0.0006,0.0001,0.0010,,0.0009,,"[0.97770, 0.97772, 0.97768]",,0.0006
4,750004,,0.0091,0.0012,0.0107,,0.0095,,"[0.97770, 0.97772, 0.97768]",,0.0078


,id,y
0,750000,0.00635
1,750001,0.06739
2,750002,0.00067
3,750003,0.00059
4,750004,0.00804
...,...,...
249995,999995,0.00060
249996,999996,0.04145
249997,999997,0.60053
249998,999998,0.00081


CPU times: user 2min 53s, sys: 6.26 s, total: 3min
Wall time: 2min 55s
